<a href="https://colab.research.google.com/github/Mayureshnema04/Mayur786/blob/main/llm_evaluation_metrics2_simplified.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Evaluation Metrics — from scratch

**Hands-on session: "Evaluating LLMs: benchmarks, leaderboards and metrics".**

The idea of this notebook: **write every metric by hand first, then check it against the
standard library.** A metric you have written yourself is a metric you can no longer misread.

| Section | Metric | Checked against |
|---|---|---|
| 1 | Setup (install, model, helpers) | – |
| 2 | Perplexity | computed from model output |
| 3 | BLEU | `sacrebleu` |
| 4 | ROUGE-1 / -2 / -L | `rouge_score` |
| 5 | Accuracy, precision, recall, F1 | `scikit-learn` |
| 6 | Exact match, token-F1 | the SQuAD convention |
| 7 | Speed and memory | measured on a real model |
| 8 | Scoring real model answers | – |
| 9 | LLM-as-judge | – |

Each section ends with **a case where the metric lies to you**. Those demos are the real
lesson: they stop you from reporting a number that means nothing.

> **Not a programmer? No problem.** You do not need to understand every line of code.
> Run each cell (Shift + Enter), then read the *printed output* and the comments that start
> with `#`. Every code cell begins with three lines: **WHAT** it does, its **INPUT**, and its **OUTPUT**.

Everything runs free on a normal Colab CPU runtime (a GPU just makes Section 7 faster).

## Section 1 — Setup

Four small packages do the work:

* `sacrebleu` and `rouge-score` — the *reference* BLEU / ROUGE tools we compare our own code against
* `scikit-learn` — reference classification metrics
* `transformers` + `torch` — to load and run a small language model

For real projects you would use `lighteval` (Hugging Face) or `lm-evaluation-harness`
(EleutherAI). Here we stay at the metric level, because that is where people make mistakes.

In [1]:
# WHAT IT DOES : installs the Python packages this notebook needs (takes 1-2 minutes the first time)
# INPUT        : nothing
# OUTPUT       : a short confirmation message (pip itself stays quiet because of -q)

%pip install -q "transformers>=5.0" torch sacrebleu rouge-score scikit-learn pandas

print("Install finished. If Colab asks you to restart the runtime, do it, then run from the next cell.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 6.8 MB/s eta 0:00:00
Install finished. If Colab asks you to restart the runtime, do it, then run from the next cell.


In [2]:
# WHAT IT DOES : imports the libraries we use and decides whether to run on GPU or CPU
# INPUT        : nothing
# OUTPUT       : the PyTorch version and the device name, e.g.  running on : cpu

import math                    # math.log / math.exp - needed for BLEU and perplexity
import re                      # regular expressions - used to split text into words
import time                    # stopwatch - used to measure speed in Section 7
import warnings                # lets us hide harmless warning messages
from collections import Counter  # counts how many times each item appears (e.g. each word)

import torch                   # the deep-learning library that runs the language model
import pandas as pd           # tables (DataFrames) - used to display results neatly

warnings.filterwarnings("ignore")   # keep the output clean for the session

# Use the GPU if there is one, otherwise the CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("torch      :", torch.__version__)
print("running on :", DEVICE)

torch      : 2.11.0+cpu
running on : cpu


### Two tiny text helpers (used by BLEU and ROUGE)

BLEU and ROUGE count **overlapping words**, so *how* you split text into words changes the score.
This is why published BLEU numbers were once not comparable between papers.

We use one simple rule everywhere: **lowercase, keep only letters and digits**.
Our code and the libraries then see exactly the same words, so the comparison is fair.

An **n-gram** is a run of *n* neighbouring words: 1-gram = one word, 2-gram = a pair of words, and so on.

In [3]:
# WHAT IT DOES : defines two helpers
#                tokenize(text)    -> list of lowercase words
#                ngrams(words, n)  -> count of every run of n neighbouring words
# INPUT        : a sentence (text) / a list of words and the number n
# OUTPUT       : the two print() lines below show an example of each

def tokenize(text):
    # find every run of letters/digits in the lowercase text
    return re.findall(r"[a-z0-9]+", text.lower())

def ngrams(words, n):
    # slide a window of size n over the word list and count each window
    return Counter(tuple(words[i:i + n]) for i in range(len(words) - n + 1))

print(tokenize("Refunds are issued within 5 business days."))
# -> ['refunds', 'are', 'issued', 'within', '5', 'business', 'days']

print(dict(ngrams(tokenize("the cat sat on the mat"), 2)))
# -> {('the','cat'): 1, ('cat','sat'): 1, ('sat','on'): 1, ('on','the'): 1, ('the','mat'): 1}

['refunds', 'are', 'issued', 'within', '5', 'business', 'days']
{('the', 'cat'): 1, ('cat', 'sat'): 1, ('sat', 'on'): 1, ('on', 'the'): 1, ('the', 'mat'): 1}


### Load a small model to produce real text

We want to measure *real* model output, not only made-up examples.
`Qwen2.5-0.5B-Instruct` is small (about 1 GB download), free to use, and needs no login.

In [4]:
# WHAT IT DOES : downloads (first time only) and loads the small language model + its tokenizer
#                The "tokenizer" turns text into numbers (tokens) the model understands, and back.
# INPUT        : the model name on Hugging Face
# OUTPUT       : model size, e.g.  Qwen/Qwen2.5-0.5B-Instruct : 0.49 billion parameters

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# float16 on GPU is faster; float32 on CPU is the safe choice
dtype = torch.float16 if DEVICE == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(DEVICE)
model.eval()                       # "evaluation mode": we are only using the model, not training it

n_params = sum(p.numel() for p in model.parameters())   # total number of weights
print(f"{MODEL_NAME} : {n_params / 1e9:.2f} billion parameters")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-0.5B-Instruct : 0.49 billion parameters


In [5]:
# WHAT IT DOES : defines chat(), one helper that asks the model a question and returns its answer.
#                Sections 7, 8 and 9 all reuse it.
# INPUT        : prompt (your question), system (instructions on how to answer), max_new_tokens (answer length limit)
# OUTPUT       : the model's answer as a string
#                (nothing is printed here - the function is only defined)

def to_model_input(prompt, system):
    # wrap the question in the chat format the model was trained on, then convert to tokens
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return tokenizer(text, return_tensors="pt").to(DEVICE)

@torch.inference_mode()            # tells PyTorch "no training": faster and uses less memory
def chat(prompt, system="Answer in one short sentence. Be factual.", max_new_tokens=60):
    inputs = to_model_input(prompt, system)
    # do_sample=False -> always pick the most likely next word, so results are repeatable
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]       # keep only the newly written part
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(chat("What is the capital of France?"))     # quick test

The capital of France is Paris.


---
## Section 2 — Perplexity

**In one sentence:** perplexity is roughly *"how many equally likely words was the model choosing
between at each step?"* **Lower is better** (the model was less surprised).

Under the hood it is the exponential of the average "surprise" (cross-entropy loss) per token:

$$\text{PPL} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(t_i \mid t_{<i})\right)$$

When we give the model the text as its own `labels`, it computes that loss for us.

In [6]:
# WHAT IT DOES : defines perplexity(text) and scores four sentences, from natural to random
# INPUT        : a piece of text
# OUTPUT       : one perplexity number per sentence (lower = the model finds it more natural)
#                Natural English is lowest by a wide margin; shuffled and nonsense text are far higher.

@torch.inference_mode()
def perplexity(text):
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)       # text -> tokens
    result = model(**inputs, labels=inputs["input_ids"])           # model returns the average loss
    return torch.exp(result.loss).item()                           # perplexity = e ^ loss

samples = {
    "natural English":      "The company issues refunds to the original payment method within five business days.",
    "same words, shuffled": "days business five within method payment original the to refunds issues company The",
    "grammatical nonsense": "The purple refund issued seventeen bicycles into the payment of Tuesday quietly.",
    "random characters":    "qx zvbn plkj wroi mnbv asdf ghjk qpwo eirut yalskd jfhgz",
}

for label, text in samples.items():
    print(f"{label:<22} perplexity = {perplexity(text):8.2f}")

natural English        perplexity =   103.81
same words, shuffled   perplexity = 14647.83
grammatical nonsense   perplexity =  8792.05
random characters      perplexity =  1270.54


Look at row 2: it has **exactly the same words** as row 1, just in a different order, and
perplexity jumps. So perplexity measures **fluency and structure**, not vocabulary.

### Where perplexity lies to you

Perplexity measures how **confident** the model is, not whether the text is **true**.
A fluent, confident falsehood scores as well as a fluent truth — and that is exactly what a
hallucination is.

In [7]:
# WHAT IT DOES : compares perplexity of a TRUE sentence and a FALSE one (a very common myth)
# INPUT        : two sentences about the brain
# OUTPUT       : two perplexity values. The FALSE myth gets the LOWER (better-looking) perplexity,
#                because the model has seen that myth many times and finds it familiar.

true_claim  = "Humans use all parts of their brain."
false_claim = "Humans only use ten percent of their brain."

print(f"TRUE  statement   perplexity = {perplexity(true_claim):7.2f}")
print(f"FALSE statement   perplexity = {perplexity(false_claim):7.2f}")
print("\nIf perplexity could detect falsehood, the false statement would score clearly worse.")
print("A familiar myth looks MORE natural to the model than the truth. Never use perplexity as a hallucination detector.")

TRUE  statement   perplexity =   88.35
FALSE statement   perplexity =   30.31

If perplexity could detect falsehood, the false statement would score clearly worse.
A familiar myth looks MORE natural to the model than the truth. Never use perplexity as a hallucination detector.


**The second trap: tokenisers.** Perplexity is measured *per token*, and different model families
cut the same sentence into different numbers of tokens. A model with a coarser tokeniser can show
a higher per-token perplexity on identical text without being any worse.

So compare perplexity **within one model family only**, or convert to *bits per character*
(below), which does not depend on the tokeniser.

In [8]:
# WHAT IT DOES : shows tokens vs characters for one sentence and converts perplexity to bits per character
# INPUT        : the "natural English" sentence from the earlier cell
# OUTPUT       : token count, character count, perplexity, and bits per character
#                (bits per character is the number you may compare across different models)

text     = samples["natural English"]
n_tokens = len(tokenizer(text)["input_ids"])    # how many tokens the model sees
n_chars  = len(text)                            # how many characters the text has
ppl      = perplexity(text)

print(f"tokens                 : {n_tokens}")
print(f"characters             : {n_chars}")
print(f"perplexity (per token) : {ppl:.2f}")
# log2(ppl) = bits per token;  x tokens / characters = bits per character
print(f"bits per character     : {math.log2(ppl) * n_tokens / n_chars:.3f}   <- comparable across tokenisers")

tokens                 : 14
characters             : 84
perplexity (per token) : 103.81
bits per character     : 1.116   <- comparable across tokenisers


---
## Section 3 — BLEU

BLEU asks: **"of the word-groups you produced, how many appear in the reference answer?"**
That is a *precision* question, which suits translation, where saying something that is not in
the reference is usually a mistake.

BLEU is built from four ideas. We add them one at a time:

1. **n-gram precision** — fraction of your 1-, 2-, 3-, 4-word groups found in the reference
2. **clipping** — you cannot score extra by repeating a word more often than the reference does
3. **geometric mean over n = 1..4** — punishes wrong word order
4. **brevity penalty** — punishes answers that are too short

### 3.1 Why clipping exists

Without clipping, a "translation" made of one repeated word could get a perfect score.

In [9]:
# WHAT IT DOES : compares NAIVE precision with CLIPPED precision for a cheating candidate
# INPUT        : reference  = "the cat sat on the mat"
#                candidate  = "the the the the the the"  (the same word, six times)
# OUTPUT       : naive precision 1.000 (looks perfect)  vs  clipped precision 0.333 (fair)

reference = "the cat sat on the mat"
cheat     = "the the the the the the"

ref_counts  = ngrams(tokenize(reference), 1)     # {'the': 2, 'cat': 1, 'sat': 1, 'on': 1, 'mat': 1}
cand_counts = ngrams(tokenize(cheat), 1)         # {'the': 6}
cand_total  = sum(cand_counts.values())          # 6 words in the candidate

# naive: count every candidate word that appears anywhere in the reference
naive   = sum(c for g, c in cand_counts.items() if g in ref_counts) / cand_total
# clipped: a word can count at most as many times as it appears in the reference
clipped = sum(min(c, ref_counts[g]) for g, c in cand_counts.items()) / cand_total

print(f"candidate : {cheat!r}")
print(f"reference : {reference!r}\n")
print(f"naive   1-gram precision = {naive:.3f}   <- every word IS in the reference")
print(f"clipped 1-gram precision = {clipped:.3f}   <- 'the' is in the reference twice, so only 2 of 6 count")

candidate : 'the the the the the the'
reference : 'the cat sat on the mat'

naive   1-gram precision = 1.000   <- every word IS in the reference
clipped 1-gram precision = 0.333   <- 'the' is in the reference twice, so only 2 of 6 count


### 3.2 The full BLEU function

This is **corpus-level** BLEU: add up the matches over *all* sentences first, then divide.
Do not average per-sentence BLEU scores — one short sentence with no matching 4-gram scores
zero and ruins the average.

In [10]:
# WHAT IT DOES : defines bleu(candidates, references) and runs it on three example sentences
# INPUT        : candidates = list of model sentences, references = list of correct sentences (same order)
# OUTPUT       : precision for 1,2,3,4-grams, lengths, brevity penalty, and the final BLEU score (0-100)
#                Higher is better; 100 means identical text.

def bleu(candidates, references, max_n=4, verbose=False):
    matches = [0] * max_n        # matches[k] = clipped matching (k+1)-grams, over all sentences
    totals  = [0] * max_n        # totals[k]  = all (k+1)-grams the candidates produced
    cand_len = ref_len = 0       # total words in candidates / references

    for cand, ref in zip(candidates, references):
        c_words, r_words = tokenize(cand), tokenize(ref)
        cand_len += len(c_words)
        ref_len  += len(r_words)
        for n in range(1, max_n + 1):
            c_grams, r_grams = ngrams(c_words, n), ngrams(r_words, n)
            matches[n - 1] += sum(min(count, r_grams[g]) for g, count in c_grams.items())   # clipping
            totals[n - 1]  += sum(c_grams.values())

    precisions = [m / t if t else 0.0 for m, t in zip(matches, totals)]

    # geometric mean of the four precisions (it is 0 if any single precision is 0)
    if min(precisions) == 0:
        geo_mean = 0.0
    else:
        geo_mean = math.exp(sum(math.log(p) for p in precisions) / max_n)

    # brevity penalty: 1.0 if the candidates are long enough, otherwise it shrinks the score
    brevity = 1.0 if cand_len > ref_len else math.exp(1 - ref_len / max(cand_len, 1))
    score = 100 * brevity * geo_mean

    if verbose:
        for n, p in enumerate(precisions, 1):
            print(f"  {n}-gram precision = {p:.4f}   ({matches[n-1]}/{totals[n-1]})")
        print(f"  candidate length {cand_len}, reference length {ref_len}")
        print(f"  brevity penalty  = {brevity:.4f}")
        print(f"  BLEU             = {score:.4f}")
    return score

candidates = [
    "the cat sat on a mat",
    "there is a cat on the mat",
    "refunds are issued within five business days",
]
references = [
    "the cat sat on the mat",
    "the cat is on the mat",
    "refunds are issued within five business days",
]

our_bleu = bleu(candidates, references, verbose=True)

  1-gram precision = 0.8500   (17/20)
  2-gram precision = 0.6471   (11/17)
  3-gram precision = 0.5714   (8/14)
  4-gram precision = 0.4545   (5/11)
  candidate length 20, reference length 19
  brevity penalty  = 1.0000
  BLEU             = 61.4788


### 3.3 Check against `sacrebleu`

We give `sacrebleu` the same pre-split words and switch off its own tokenising and smoothing,
so both implementations do genuinely the same arithmetic. The scores should match.

In [11]:
# WHAT IT DOES : compares our BLEU score with the official sacrebleu library
# INPUT        : the same candidates / references as above
# OUTPUT       : both scores and "match : True" if they agree

import sacrebleu       # the standard reference implementation of BLEU

# give sacrebleu the same cleaned-up words that our own function uses
clean_cands = [" ".join(tokenize(c)) for c in candidates]
clean_refs  = [" ".join(tokenize(r)) for r in references]

library = sacrebleu.BLEU(tokenize="none", smooth_method="none").corpus_score(clean_cands, [clean_refs])

print(f"our implementation : {our_bleu:.4f}")
print(f"sacrebleu          : {library.score:.4f}")
print(f"match              : {abs(our_bleu - library.score) < 1e-6}")

our implementation : 61.4788
sacrebleu          : 61.4788
match              : True


### 3.4 Where BLEU lies to you

The most important cell in this section. A **correct paraphrase** shares almost no word-groups
with the reference and scores zero, while a **fluent falsehood** that reuses the reference's
words scores higher.

In [12]:
# WHAT IT DOES : scores five different answers against one reference sentence
# INPUT        : one reference + five candidate answers (some correct, some wrong)
# OUTPUT       : one BLEU score per candidate.
#                The correct paraphrases score ~0; the FALSE sentence scores higher than both.

reference = "refunds are issued within five business days"

cases = {
    "identical":                 "refunds are issued within five business days",
    "correct paraphrase":        "reimbursements are provided within about a week",
    "correct, reworded":         "we send your money back in five working days",
    "FALSE but same vocabulary": "refunds are issued within fifty business days",
    "word salad from reference": "days business five within issued are refunds",
}

print(f"reference: {reference!r}\n")
for label, cand in cases.items():
    print(f"  BLEU = {bleu([cand], [reference]):6.2f}   {label:<26} {cand!r}")

print("\nBLEU measures surface word overlap. It has no understanding of meaning.")

reference: 'refunds are issued within five business days'

  BLEU = 100.00   identical                  'refunds are issued within five business days'
  BLEU =   0.00   correct paraphrase         'reimbursements are provided within about a week'
  BLEU =   0.00   correct, reworded          'we send your money back in five working days'
  BLEU =  48.89   FALSE but same vocabulary  'refunds are issued within fifty business days'
  BLEU =   0.00   word salad from reference  'days business five within issued are refunds'

BLEU measures surface word overlap. It has no understanding of meaning.


---
## Section 4 — ROUGE (the mirror image of BLEU)

* BLEU asks: *of what **you said**, how much was in the reference?* → **precision**
* ROUGE asks: *of what was **in the reference**, how much did you say?* → **recall**

That is why summarisation uses ROUGE: a summary's typical failure is **leaving things out**.

Three common variants:

* **ROUGE-1** — single words. Ignores word order.
* **ROUGE-2** — pairs of neighbouring words. Adds a little fluency.
* **ROUGE-L** — longest common subsequence (words in the same order, gaps allowed).

Each gives **P** (precision), **R** (recall) and **F** (F-score, the balance of the two).

In [13]:
# WHAT IT DOES : defines ROUGE-1, ROUGE-2 and ROUGE-L, then scores one example prediction
# INPUT        : a reference sentence and a predicted sentence
# OUTPUT       : P, R, F (each 0-1, higher is better) for rouge1, rouge2, rougeL

def precision_recall_f(matches, pred_total, ref_total):
    # turn match counts into precision, recall and F-score
    p = matches / pred_total if pred_total else 0.0
    r = matches / ref_total  if ref_total  else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f

def rouge_n(reference, prediction, n):
    # ROUGE-1 (n=1) and ROUGE-2 (n=2): overlap of n-grams
    ref_grams  = ngrams(tokenize(reference), n)
    pred_grams = ngrams(tokenize(prediction), n)
    matches = sum(min(c, ref_grams[g]) for g, c in pred_grams.items())
    return precision_recall_f(matches, sum(pred_grams.values()), sum(ref_grams.values()))

def lcs_length(a, b):
    # length of the longest common subsequence of word lists a and b
    # table[i][j] = answer for the first i words of a and the first j words of b
    table = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                table[i][j] = table[i - 1][j - 1] + 1                       # words match: extend
            else:
                table[i][j] = max(table[i - 1][j], table[i][j - 1])        # no match: keep the best so far
    return table[-1][-1]

def rouge_l(reference, prediction):
    ref_words, pred_words = tokenize(reference), tokenize(prediction)
    return precision_recall_f(lcs_length(ref_words, pred_words), len(pred_words), len(ref_words))

REF  = "the company issues refunds to the original payment method within five business days"
PRED = "refunds are issued to the original payment method in five business days"

print(f"reference : {REF!r}")
print(f"prediction: {PRED!r}\n")
for name, (p, r, f) in [("rouge1", rouge_n(REF, PRED, 1)),
                        ("rouge2", rouge_n(REF, PRED, 2)),
                        ("rougeL", rouge_l(REF, PRED))]:
    print(f"  {name}   P={p:.4f}  R={r:.4f}  F={f:.4f}")

reference : 'the company issues refunds to the original payment method within five business days'
prediction: 'refunds are issued to the original payment method in five business days'

  rouge1   P=0.7500  R=0.6923  F=0.7200
  rouge2   P=0.5455  R=0.5000  F=0.5217
  rougeL   P=0.7500  R=0.6923  F=0.7200


In [14]:
# WHAT IT DOES : checks our ROUGE numbers against the official rouge_score library
# INPUT        : the same REF and PRED sentences
# OUTPUT       : one line per variant with our F-score, the library's F-score and "match=True"

from rouge_score import rouge_scorer      # the standard reference implementation of ROUGE

scorer  = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
library = scorer.score(REF, PRED)

ours = {"rouge1": rouge_n(REF, PRED, 1), "rouge2": rouge_n(REF, PRED, 2), "rougeL": rouge_l(REF, PRED)}

for name, (p, r, f) in ours.items():
    lib = library[name]
    same = all(abs(a - b) < 1e-6 for a, b in zip((p, r, f), (lib.precision, lib.recall, lib.fmeasure)))
    print(f"  {name}   ours F={f:.4f}   library F={lib.fmeasure:.4f}   match={same}")

  rouge1   ours F=0.7200   library F=0.7200   match=True
  rouge2   ours F=0.5217   library F=0.5217   match=True
  rougeL   ours F=0.7200   library F=0.7200   match=True


### Where ROUGE lies to you

Two common failures in real summarisation projects:

1. **A reversed meaning gets rewarded.** Adding one word — *not* — flips the meaning, yet the
   sentence is still a near-copy of the gold summary, so its ROUGE score goes **up**.
2. **Copying beats summarising.** Pasting the source text scores well, even though it adds nothing.

In [15]:
# WHAT IT DOES : scores three candidate summaries against one gold (ideal) summary
# INPUT        : a source text, a gold summary, three candidate summaries
# OUTPUT       : ROUGE-1 F and ROUGE-L F for each candidate.
#                The INVERTED summary (meaning reversed!) scores highest.

source = ("The refund policy applies to all hardware purchased directly from us. "
          "Refunds are issued within five business days of approval.")
gold   = "Hardware bought directly is refundable, with money returned in five business days."

candidates = {
    "good abstractive summary": "Direct hardware purchases can be refunded within five working days.",
    "verbatim copy of source":  source,
    "FACTUALLY INVERTED":       "Hardware bought directly is not refundable, with money returned in five business days.",
}

print(f"gold summary: {gold!r}\n")
for label, cand in candidates.items():
    r1 = rouge_n(gold, cand, 1)[2]      # [2] = the F-score
    rl = rouge_l(gold, cand)[2]
    print(f"  ROUGE-1 F={r1:.3f}   ROUGE-L F={rl:.3f}   {label}")

print("\nOne word reversed the meaning and the score went UP.")
print("ROUGE measures word overlap. It cannot see whether the summary is faithful.")

gold summary: 'Hardware bought directly is refundable, with money returned in five business days.'

  ROUGE-1 F=0.273   ROUGE-L F=0.273   good abstractive summary
  ROUGE-1 F=0.312   ROUGE-L F=0.312   verbatim copy of source
  ROUGE-1 F=0.960   ROUGE-L F=0.960   FACTUALLY INVERTED

One word reversed the meaning and the score went UP.
ROUGE measures word overlap. It cannot see whether the summary is faithful.


---
## Section 5 — Accuracy, precision, recall and F1

For a yes/no decision there are exactly four outcomes. Every metric here is a different ratio of them.

|  | actually yes | actually no |
|---|---|---|
| **predicted yes** | TP (true positive) | FP (false positive) |
| **predicted no**  | FN (false negative) | TN (true negative) |

* **Accuracy** = (TP + TN) / all — how often was it right overall?
* **Precision** = TP / (TP + FP) — when it said yes, how often was it right?
* **Recall** = TP / (TP + FN) — of all the real yeses, how many did it find?
* **F1** — balance of precision and recall; dragged down by whichever is worse.

In [16]:
# WHAT IT DOES : defines metrics() which counts TP/FP/FN/TN and computes the four scores
# INPUT        : y_true = correct answers (1 = yes, 0 = no), y_pred = model's answers
# OUTPUT       : TP, FP, FN, TN counts and accuracy, precision, recall, f1 (0-1, higher is better)

def metrics(y_true, y_pred):
    pairs = list(zip(y_true, y_pred))
    tp = sum(t == 1 and p == 1 for t, p in pairs)    # said yes, really yes
    fp = sum(t == 0 and p == 1 for t, p in pairs)    # said yes, really no  (false alarm)
    fn = sum(t == 1 and p == 0 for t, p in pairs)    # said no,  really yes (missed)
    tn = sum(t == 0 and p == 0 for t, p in pairs)    # said no,  really no

    accuracy  = (tp + tn) / len(pairs)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

# 20 example decisions: the truth, and what a model said
y_true = [1,0,1,1,0,1,0,0,1,0,1,0,1,1,0,0,1,0,0,1]
y_pred = [1,0,1,0,0,1,1,0,1,0,0,0,1,1,1,0,1,0,0,1]

ours = metrics(y_true, y_pred)
for name, value in ours.items():
    print(f"  {name:<10} {value:.4f}" if isinstance(value, float) else f"  {name:<10} {value}")

  TP         8
  FP         2
  FN         2
  TN         8
  accuracy   0.8000
  precision  0.8000
  recall     0.8000
  f1         0.8000


In [17]:
# WHAT IT DOES : checks our numbers against scikit-learn, the standard machine-learning library
# INPUT        : the same y_true / y_pred lists
# OUTPUT       : our value vs scikit-learn's value for each metric - they should be identical

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

sk_p, sk_r, sk_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)

print(f"  accuracy   ours {ours['accuracy']:.4f}   sklearn {accuracy_score(y_true, y_pred):.4f}")
print(f"  precision  ours {ours['precision']:.4f}   sklearn {sk_p:.4f}")
print(f"  recall     ours {ours['recall']:.4f}   sklearn {sk_r:.4f}")
print(f"  f1         ours {ours['f1']:.4f}   sklearn {sk_f1:.4f}")

  accuracy   ours 0.8000   sklearn 0.8000
  precision  ours 0.8000   sklearn 0.8000
  recall     ours 0.8000   sklearn 0.8000
  f1         ours 0.8000   sklearn 0.8000


### Where accuracy lies to you — imbalanced data

This is the most common reporting mistake in applied machine learning. If 1 ticket in 100 is
fraud, a model that always answers "not fraud" gets **99% accuracy** and catches **zero** fraud.

In [18]:
# WHAT IT DOES : compares a "lazy" model (always says no) with a model that really tries
# INPUT        : 1000 tickets, of which only 10 are fraud
# OUTPUT       : accuracy, precision, recall and F1 for both models.
#                The lazy model's accuracy (0.990) looks almost as good as the real model's, yet its recall is 0.

y_true_fraud = [1] * 10 + [0] * 990                         # 10 fraud, 990 normal
lazy_pred    = [0] * 1000                                   # always "not fraud"
real_pred    = [1] * 7 + [0] * 3 + [0] * 985 + [1] * 5      # finds 7 of 10 frauds, 5 false alarms

for label, pred in [("lazy model (always 'no')", lazy_pred), ("real model", real_pred)]:
    r = metrics(y_true_fraud, pred)
    print(f"{label}")
    print(f"   accuracy  {r['accuracy']:.3f}")
    print(f"   precision {r['precision']:.3f}   recall {r['recall']:.3f}   F1 {r['f1']:.3f}\n")

print("The lazy model scores 99% accuracy and catches ZERO fraud. Recall exposes it.")
print("Never report accuracy alone on imbalanced data.")

lazy model (always 'no')
   accuracy  0.990
   precision 0.000   recall 0.000   F1 0.000

real model
   accuracy  0.992
   precision 0.583   recall 0.700   F1 0.636

The lazy model scores 99% accuracy and catches ZERO fraud. Recall exposes it.
Never report accuracy alone on imbalanced data.


### Macro vs micro averaging

With more than two classes you must say *how* you averaged:

* **macro** — compute the metric for each class, then average. Every class counts equally,
  so a small class matters as much as a big one.
* **micro** — pool all decisions first. Big classes dominate. (For single-label tasks, micro-F1 equals accuracy.)

In [19]:
# WHAT IT DOES : scores a 3-class ticket-routing task with micro, macro and weighted averages
# INPUT        : 100 tickets: 60 billing, 30 shipping, 10 legal. The model is bad at the rare 'legal' class.
# OUTPUT       : P / R / F1 for each averaging style, then a per-class report.
#                Macro-F1 is much lower than micro-F1 because 'legal' is mostly missed.

y_t = ["billing"] * 60 + ["shipping"] * 30 + ["legal"] * 10
y_p = (["billing"] * 58 + ["shipping"] * 2          # 60 billing: 58 right, 2 sent to shipping
       + ["shipping"] * 28 + ["billing"] * 2        # 30 shipping: 28 right, 2 sent to billing
       + ["billing"] * 8 + ["legal"] * 2)           # 10 legal: only 2 right, 8 sent to billing

for avg in ["micro", "macro", "weighted"]:
    p, r, f, _ = precision_recall_fscore_support(y_t, y_p, average=avg, zero_division=0)
    print(f"  {avg:<9} P={p:.3f}  R={r:.3f}  F1={f:.3f}")

print("\nPer-class report - always look at this before trusting an average:\n")
print(classification_report(y_t, y_p, zero_division=0))

  micro     P=0.880  R=0.880  F1=0.880
  macro     P=0.929  R=0.700  F1=0.724
  weighted  P=0.892  R=0.880  F1=0.857

Per-class report - always look at this before trusting an average:

              precision    recall  f1-score   support

     billing       0.85      0.97      0.91        60
       legal       1.00      0.20      0.33        10
    shipping       0.93      0.93      0.93        30

    accuracy                           0.88       100
   macro avg       0.93      0.70      0.72       100
weighted avg       0.89      0.88      0.86       100



---
## Section 6 — Exact match and token-F1

For short-answer questions you can compare the strings directly.
**Exact match (EM)** is strict: 1 if identical, else 0. **Token-F1** gives partial credit for shared words.
The SQuAD benchmark reports both, after cleaning the text (lowercase, no punctuation, no "a/an/the").

In [20]:
# WHAT IT DOES : defines exact_match() and token_f1(), then scores five answers against "5 business days"
# INPUT        : a predicted answer and the gold (correct) answer
# OUTPUT       : a table with EM (0 or 1) and token-F1 (0-1) for each answer

def normalize(text):
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)     # remove the articles a / an / the
    text = re.sub(r"[^a-z0-9 ]", " ", text)         # remove punctuation
    return " ".join(text.split())                   # collapse extra spaces

def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))

def token_f1(pred, gold):
    pred_words, gold_words = normalize(pred).split(), normalize(gold).split()
    shared = sum((Counter(pred_words) & Counter(gold_words)).values())   # words in common
    if shared == 0:
        return 0.0
    precision, recall = shared / len(pred_words), shared / len(gold_words)
    return 2 * precision * recall / (precision + recall)

gold  = "5 business days"
preds = ["5 business days", "five business days", "The answer is 5 business days.",
         "5 days", "about a week"]

print(f"gold: {gold!r}\n")
print(f"{'prediction':<34} {'EM':>5} {'token-F1':>9}")
for pred in preds:
    print(f"  {pred!r:<32} {exact_match(pred, gold):>5.0f} {token_f1(pred, gold):>9.3f}")

print("\nRow 2: 'five' means '5' and is correct, yet it scores 0 on both.")
print("How you clean the text is part of your metric. Write it down.")

gold: '5 business days'

prediction                            EM  token-F1
  '5 business days'                    1     1.000
  'five business days'                 0     0.667
  'The answer is 5 business days.'     0     0.750
  '5 days'                             0     0.800
  'about a week'                       0     0.000

Row 2: 'five' means '5' and is correct, yet it scores 0 on both.
How you clean the text is part of your metric. Write it down.


---
## Section 7 — Speed and memory: what users actually feel

None of the quality metrics tell you whether a system is *usable*. These do:

* **TTFT** (time to first token) — how long the user waits before anything appears
* **Throughput** — tokens produced per second; drives capacity and cost
* **Memory** — decides which hardware you need to buy

**Warm up before timing.** The very first call includes one-time setup work, so it is slower and
not representative. Timing it is the most common benchmarking mistake.

In [21]:
# WHAT IT DOES : defines measure() which times the model, then runs 1 cold + 3 warm measurements
# INPUT        : a prompt and a maximum number of new tokens
# OUTPUT       : TTFT (ms) and tokens/second for the cold run, each warm run, and the warm average.
#                The cold run is often a little slower than the warm runs (on a GPU the gap is much larger).

def stopwatch(inputs, new_tokens):
    # run the model for `new_tokens` tokens and return (output, seconds taken)
    start = time.perf_counter()
    output = model.generate(**inputs, max_new_tokens=new_tokens, do_sample=False)
    if DEVICE == "cuda":
        torch.cuda.synchronize()        # wait for the GPU to finish before stopping the clock
    return output, time.perf_counter() - start

@torch.inference_mode()
def measure(prompt, max_new_tokens=64):
    inputs = to_model_input(prompt, "You are a helpful assistant.")
    prompt_tokens = inputs["input_ids"].shape[-1]

    _, ttft = stopwatch(inputs, 1)                          # time to produce just ONE token
    output, total = stopwatch(inputs, max_new_tokens)       # time for the full answer
    new_tokens = output.shape[-1] - prompt_tokens

    return {"prompt_tokens": prompt_tokens, "new_tokens": new_tokens,
            "ttft_ms": ttft * 1000, "tokens_per_s": new_tokens / total}

PROMPT = "Explain in three sentences why evaluation matters when choosing a language model."

cold = measure(PROMPT)                         # first call: includes one-time setup
warm_runs = [measure(PROMPT) for _ in range(3)]    # next calls: representative

print(f"COLD run     TTFT {cold['ttft_ms']:7.1f} ms   {cold['tokens_per_s']:6.1f} tokens/s")
for i, run in enumerate(warm_runs, 1):
    print(f"warm run {i}   TTFT {run['ttft_ms']:7.1f} ms   {run['tokens_per_s']:6.1f} tokens/s")

warm = pd.DataFrame(warm_runs).mean()          # average of the three warm runs
print(f"\nwarm average: TTFT {warm['ttft_ms']:.1f} ms, {warm['tokens_per_s']:.1f} tokens/s")

COLD run     TTFT  3697.0 ms      4.2 tokens/s
warm run 1   TTFT   570.5 ms      5.1 tokens/s
warm run 2   TTFT   573.0 ms      5.4 tokens/s
warm run 3   TTFT   682.2 ms      5.0 tokens/s

warm average: TTFT 608.5 ms, 5.2 tokens/s


In [22]:
# WHAT IT DOES : reports memory used by the model and estimates the cost of generating 1 million tokens
# INPUT        : the loaded model and the warm tokens/second measured above
# OUTPUT       : memory in GB, and cost per 1M output tokens (uses a made-up hourly price you can change)

weights_gb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9
print(f"model weights in memory : {weights_gb:.2f} GB")
if DEVICE == "cuda":
    print(f"peak GPU memory         : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

HOURLY_PRICE = 0.40       # example price of renting a GPU per hour - replace with your own
tokens_per_hour = warm["tokens_per_s"] * 3600
cost_per_million = HOURLY_PRICE / tokens_per_hour * 1_000_000

print(f"\nAt {warm['tokens_per_s']:.1f} tokens/s and a price of {HOURLY_PRICE}/hour:")
print(f"cost per 1 million output tokens = {cost_per_million:.2f}")
print("Compare this with an API's published price - it settles most 'self-host or not' debates.")

model weights in memory : 1.98 GB

At 5.2 tokens/s and a price of 0.4/hour:
cost per 1 million output tokens = 21.50
Compare this with an API's published price - it settles most 'self-host or not' debates.


**TTFT grows with prompt length**, because the model must read the whole prompt before it can
write anything. This is why stuffing twenty retrieved chunks into a RAG prompt makes an assistant feel slow.

In [23]:
# WHAT IT DOES : times a short prompt and a long prompt
# INPUT        : a ~10-word prompt and the same text repeated 40 times
# OUTPUT       : prompt size in tokens and TTFT for each. The long prompt has the larger TTFT.

short_prompt = "Summarise this: refunds take five days."
long_prompt  = "Summarise this: " + "Refunds are issued within five business days of approval. " * 40

for label, prompt in [("short prompt", short_prompt), ("long prompt", long_prompt)]:
    run = measure(prompt, max_new_tokens=32)
    print(f"{label:<13} {run['prompt_tokens']:>5} prompt tokens   TTFT {run['ttft_ms']:7.1f} ms")

short prompt     29 prompt tokens   TTFT   546.9 ms
long prompt     465 prompt tokens   TTFT  5020.9 ms


---
## Section 8 — Putting it together: score real model answers

Now we use the metrics on **real** answers from the model. This is what your own evaluation looks
like: a few questions, a reference answer for each, and one or more metrics suited to the task.

In [24]:
# WHAT IT DOES : asks the model 5 questions and scores each answer with BLEU, ROUGE-L, token-F1 and exact match
# INPUT        : EVAL_SET = list of (question, correct reference answer)
# OUTPUT       : a table with the model's answer and four scores per question.
#                The columns disagree - that disagreement is the lesson.

EVAL_SET = [
    ("What is the capital of France?",          "Paris is the capital of France."),
    ("How many continents are there?",          "There are seven continents."),
    ("What does CPU stand for?",                "CPU stands for central processing unit."),
    ("Who wrote the play Hamlet?",              "Hamlet was written by William Shakespeare."),
    ("What is the chemical symbol for gold?",   "The chemical symbol for gold is Au."),
]

rows = []
for question, gold in EVAL_SET:
    answer = chat(question)                                   # the model's answer
    rows.append({
        "question":  question,
        "answer":    answer[:50],
        "BLEU":      round(bleu([answer], [gold]), 1),
        "ROUGE-L":   round(rouge_l(gold, answer)[2], 3),      # [2] = F-score
        "token-F1":  round(token_f1(answer, gold), 3),
        "EM":        exact_match(answer, gold),
    })

pd.set_option("display.max_colwidth", 52)
pd.set_option("display.width", 200)
results = pd.DataFrame(rows)
results

,question,answer,BLEU,ROUGE-L,token-F1,EM
0,What is the capital of France?,The capital of France is Paris.,56.2,0.667,1.000,0.0
1,How many continents are there?,"There are 5 continents: Africa, Antarctica, Asia,",0.0,0.400,0.400,0.0
2,What does CPU stand for?,CPU stands for Central Processing Unit.,100.0,1.000,1.000,1.0
3,Who wrote the play Hamlet?,"William Shakespeare wrote the play ""Hamlet.""",0.0,0.333,0.545,0.0
4,What is the chemical symbol for gold?,The chemical symbol for gold is Au.,100.0,1.000,1.000,1.0


Look at how the columns disagree. An answer that is **correct but worded differently** gets a low BLEU
but a decent token-F1. Exact match is almost always 0 for a chatty model, so you either force a
strict answer format or use a softer metric.

**No single number is "the score".** Pick the metric that fits your task, and say which one you used.

### 🧪 Try it yourself — build your own evaluation set

Replace `EVAL_SET` with 10–20 questions from your own work and run the cell again.

* Include awkward real inputs: typos, ambiguous wording, and questions whose right answer is "I don't know".
* Write the correct answer **before** you look at the model's answer.
* Twenty examples catch big failures; about a hundred let you compare two models.
* Keep the set. Re-run it whenever you change the prompt or the model.

---
## Section 9 — LLM-as-judge, and why a small judge fails

When there is no single right answer, overlap metrics are useless and people ask *another model*
to judge. That can work — with a **strong** judge and careful design.

Here we show the opposite on purpose: our 0.5B model is too weak to judge. We ask it to pick the
better answer twice, with the order swapped. A reliable judge picks the same *answer* both times.

In [25]:
# WHAT IT DOES : asks the small model to judge 6 pairs of (correct, wrong) answers, each pair in BOTH orders
# INPUT        : 6 questions, each with a correct answer and a wrong answer
# OUTPUT       : a table of the judge's verdicts and a score.
#                A good judge says 'A' when the correct answer is first and 'B' when it is second.
#                This small judge is inconsistent: it often gives the same letter in both orders.

JUDGE_PROMPT = (
    "You are judging two answers to a question. Reply with exactly one character: A or B.\n\n"
    "Question: {q}\n\nAnswer A: {a}\n\nAnswer B: {b}\n\nBetter answer (A or B):"
)

PAIRS = [   # (question, correct answer, wrong answer)
    ("What is the capital of France?",           "Paris is the capital of France.",          "The capital of France is Berlin."),
    ("How many days are in a week?",             "There are seven days in a week.",          "There are twelve days in a week."),
    ("Who wrote Hamlet?",                        "William Shakespeare wrote Hamlet.",        "Charles Dickens wrote Hamlet."),
    ("What is 2 + 2?",                           "2 + 2 equals 4.",                          "2 + 2 equals 5."),
    ("What is the chemical symbol for water?",   "The chemical symbol for water is H2O.",    "The chemical symbol for water is CO2."),
    ("Which planet is known as the Red Planet?", "Mars is known as the Red Planet.",         "Venus is known as the Red Planet."),
]

def judge(question, answer_a, answer_b):
    prompt = JUDGE_PROMPT.format(q=question, a=answer_a, b=answer_b)
    return chat(prompt, system="You are a strict evaluator.", max_new_tokens=4)

rows = []
for question, correct, wrong in PAIRS:
    said_1 = judge(question, correct, wrong)    # correct answer is A -> should say A
    said_2 = judge(question, wrong, correct)    # correct answer is B -> should say B
    ok = said_1.upper().startswith("A") and said_2.upper().startswith("B")
    rows.append({"question": question, "correct in A -> said": said_1,
                 "correct in B -> said": said_2, "consistent": ok})

pd.DataFrame(rows)

,question,correct in A -> said,correct in B -> said,consistent
0,What is the capital of France?,B,B,False
1,How many days are in a week?,A,A,False
2,Who wrote Hamlet?,A,B,True
3,What is 2 + 2?,A,A,False
4,What is the chemical symbol for water?,A,B,True
5,Which planet is known as the Red Planet?,A,B,True


In [26]:
# WHAT IT DOES : counts how many of the 6 pairs the judge got right in BOTH orders
# INPUT        : the table of verdicts from the cell above
# OUTPUT       : one line, e.g. "consistent and correct on 3 of 6 pairs" - roughly what coin-flipping gives

n_ok = sum(r["consistent"] for r in rows)
print(f"consistent and correct on {n_ok} of {len(rows)} pairs")
print("A judge that changes its answer when you swap the order is measuring position, not quality.")

consistent and correct on 3 of 6 pairs
A judge that changes its answer when you swap the order is measuring position, not quality.


**If you do use an LLM judge:**

* Use a **strong** model, stronger than the systems you compare.
* **Swap the order and average** — position bias is real, so run every comparison twice.
* **Watch the length** — judges tend to prefer longer answers.
* Use a **different model family** than the one being judged — models favour their own style.
* **Check against humans first** on about 50 examples. If the judge disagrees with people, it measures nothing.

Better still, **make the task checkable**: if the output is a label, a number, valid JSON, or code
that passes tests, you get a fast, free, exact metric and can skip judges entirely.

---
## Where to go next

* **Use a real harness** — `lighteval` and `lm-evaluation-harness` run standard benchmarks with the prompt formats and answer parsing already handled.
* **Try semantic metrics** — `BERTScore` compares meaning, so correct paraphrases stop scoring zero (it still cannot see reversed facts).
* **Track cost and speed next to quality** — a model 2 points better but 4× slower is often the wrong choice.
* **Keep your eval set in git** — when someone asks why you chose a model, the answer should be a file, not a memory.

### Five things to remember

1. **A benchmark judges a model; an evaluation judges your product.** Only one is your job.
2. **Every metric is a proxy.** Perplexity = confidence, not truth. BLEU/ROUGE = word overlap, not meaning. Accuracy lies on imbalanced data.
3. **Pick the metric to fit the task**, not the other way round. A wrong metric still returns a number.
4. **Warm up before timing**, and report cost per million tokens.
5. **Twenty of your own examples beat any public leaderboard.**